In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel,Field
import operator

In [2]:
load_dotenv()

True

In [3]:
model=ChatGoogleGenerativeAI(model='gemini-3.6-flash')

In [4]:
class EvaluationSchema(BaseModel):
    feedback: str= Field(description='Detailed feedbackfor the essay')
    score: int =Field(description='Score out of 10',ge=0, le=10)

In [5]:
structured_model=model.with_structured_output(EvaluationSchema)

In [6]:
essay= """Title: The Role of India in the Global Artificial Intelligence Revolution

Introduction
Artificial Intelligence (AI) has emerged as the defining technology of the 21st century, reshaping economies, industries, and social structures worldwide. In this global transformation, India occupies a unique and pivotal position. Blending a massive digital infrastructure, a vast pool of technical talent, and a distinct policy vision centered on "AI for All," India is not merely an adopter of AI technologies—it is rapidly becoming a key driver, innovator, and ethical guide in the global AI ecosystem.

1. A Massive Talent Base and Developer Ecosystem
One of India’s greatest strengths in the AI domain is its human capital. As home to one of the largest engineering and STEM workforce populations in the world, India provides a substantial portion of the global AI talent pool. Major technology companies and global research labs rely heavily on Indian engineers and data scientists. Furthermore, India’s developer community ranks among the largest globally on open-source platforms like GitHub, actively contributing to machine learning libraries, foundational models, and localized applications.

2. Digital Public Infrastructure: The Foundation for Scalable AI
India's approach to technology differs significantly from Silicon Valley's market-driven model or China's state-centric surveillance model. India has pioneered "Digital Public Infrastructure" (DPI), commonly referred to as the India Stack—encompassing Aadhaar (digital identity), UPI (digital payments), and DigiLocker. This nationwide digital framework generates huge, structured, and diverse datasets. Combined with high-speed, low-cost mobile internet, this infrastructure provides a uniquely fertile ground for deploying AI solutions at an unprecedented scale.

3. Socioeconomic Transformation: "AI for Inclusive Development"
In India, AI is increasingly leveraged to address deep-seated socioeconomic challenges rather than purely luxury or consumer-facing applications:
- Agriculture: AI-driven predictive tools help farmers monitor crop health, forecast local weather patterns, and optimize soil usage, boosting agricultural productivity.
- Healthcare: In a country with a low doctor-to-patient ratio in rural areas, AI-powered diagnostic tools enable early detection of diseases, tele-pathology, and efficient clinical decision support.
- Education and Language Inclusion: Initiatives like "BharatGen" and natural language processing (NLP) models focus on bridging the digital divide across India’s 22 constitutionally recognized languages. Multilingual AI tools allow non-English speakers to access government services, financial tools, and personalized learning materials in their native dialects.

4. Government Initiatives and Strategic Vision
Recognizing AI's strategic importance, the Government of India has taken proactive steps to foster innovation. The NITI Aayog's "National Strategy for Artificial Intelligence" laid the initial groundwork by prioritizing sectors like healthcare, agriculture, education, smart cities, and mobility. Building on this, the IndiaAI Mission—with significant funding allocated for compute power, startup incubation, and data governance—aims to establish sovereign AI computing capacity within the country. This ensures that Indian researchers and startups have the necessary high-performance compute resources to build native foundational models.

5. Global Leadership and Ethical Governance
As AI development accelerates, questions surrounding ethics, bias, data privacy, and job displacement have taken center stage. India has positioned itself as a balanced voice in global tech governance. By advocating for responsible, transparent, and democratic AI, India emphasizes that technology should serve humanity as a public good. Through international forums such as the Global Partnership on Artificial Intelligence (GPAI) and its leadership in regional summits, India advocates for equitable access to AI, ensuring that developing nations in the Global South are not left behind in the AI revolution.

Challenges Facing India's AI Ambitions
Despite its significant strengths, India faces critical challenges:
- High-Performance Compute Infrastructure: India remains dependent on foreign chipmakers and global cloud providers for high-end GPUs. Building local hardware capabilities and data centers is essential for technological sovereignty.
- Bridging the Skill Gap: While India produces millions of graduates, there remains a shortage of specialized talent in deep-tech domains like advanced machine learning research, neural architecture, and semiconductor design.
- Data Privacy and Security: Balancing rapid technological innovation with strong safeguards for citizen privacy requires constant refinement of regulatory and data protection frameworks.

Conclusion
India's role in the global AI landscape is both strategic and transformative. By leveraging its vast technical talent, robust digital public infrastructure, and a human-centric approach to innovation, India is proving that AI can be used as a force for inclusive economic growth and social empowerment. As the country moves toward its vision of becoming a developed nation ("Viksit Bharat"), its continued leadership in AI will not only shape its domestic future but will also serve as a blueprint for how emerging economies can harness cutting-edge technology for the collective good."""

In [9]:
prompt=f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10\n {essay}'

In [10]:
structured_model.invoke(prompt).feedback

'The essay demonstrates exceptional language quality with sophisticated vocabulary, flawless grammar, and clear structural organization. The tone is suitably academic and formal, utilizing appropriate technical terminology and varied sentence structures. Ideas transition smoothly from section to section, creating a persuasive and highly articulate narrative overall.'

In [ ]:
class UPSCState(TypedDict):

    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int],operator.add]
    avg_score: float

In [ ]:
def evaluate_langauage(state: UPSCState):

    prompt=f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10\n {state["essay"]}'
    output=structured_model.invoke(prompt)

    return{'language_feedback':output.feedback, 'individual_score':[output.score]}


In [ ]:
def evaluate_analysis(state: UPSCState):

    prompt=f'Evaluate the depth of anaysis of the following essay and provide a feedback and assign a score out of 10\n {state["essay"]}'
    output=structured_model.invoke(prompt)

    return{'language_feedback':output.feedback, 'individual_score':[output.score]}


In [ ]:
graph= StateGraph(UPSCState)

graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_evaluation',final_evaluation)